# GENIAC Prize 2026 テーマ 2 参加者 Notebook (2/2): Inference (推論 → 提出用 JSON)

**目的**: Base モデル (+ notebook1_sft で用意した LoRA adapter) で評価用データ 232 問を推論し、
提出用の JSON を出力する。

**このセットの構成 (2 notebook)**:
1. notebook1_sft.ipynb (adapter を用意。学習なしで参加する場合は不要)
2. **notebook2_inference.ipynb** ← 本 notebook

採点はリーダーボードで行われるため、手元で採点する notebook はない。出力した JSON を
omnicampus に提出すると自動採点され、順位が反映される。

**T4 での所要時間の目安**: 約 100 分
- Stage A (install + adapter merge + merged save)
- **Runtime restart 必須** (下記 Step 3 参照。`PATH_CHOICE="iii"` の場合は不要)
- Stage B (LLM 構築 + 232 問の推論 + JSON 出力)

Colab 無償版の GPU 利用上限は公表されておらず変動する。推論は 58 問ずつ 4 回に分けて
途中保存しているため、セッションが切れても続きから再開できる。

**前提**: `PATH_CHOICE` が `"ii"` の場合は notebook1_sft を先に完走し、adapter が
Drive (または HF Hub) に保存されていること。`"iii"` の場合は不要。

**事前に Colab の `/content/` にアップロードするファイル (配布物)**:
- `inference_config.json` — 運営指定の chat template。推論時に適用する template の正本
- `math_sft.tar.gz` — ヘルパー。評価用データの読み込みに使う

**事前に済ませておくこと**:
- Hugging Face アカウントで**評価用データの利用条件に同意**する (自動承認)
- Hugging Face の **read token を Colab Secrets に `HFH_READ` として設定**する。
  評価用データの取得に必要なので、学習せずに参加する場合も必須

**各セルの先頭には `# [名前]` を付けている**。以降の説明ではこの名前でセルを指す。


## Step 0: 環境セットアップ (uv + vLLM cu129 wheel + peft, torchao uninstall)

vLLM 0.25.1 の GitHub cu129 wheel を **uv pip 経由で直接 URL 指定**して install。

**理由**:
- PyPI 版の vllm は libcudart.so.13 preload が必須になる (rpath 情報が省略されているため)
- GitHub `+cu129` wheel は rpath 埋め込みで preload 不要 + Colab CUDA 12.8 で動作
- uv は公式 docs 推奨 (pip でも同じ URL 指定で動く)
- **torchao 0.10 (Colab pre-installed)** は peft 0.19.x の hard error 原因、uninstall で回避 (Base fp16 merge では torchao 不要)

**注意**: vllm install で transformers が 4.57.2 → 5.x に silent upgrade される。
これは推論のみなら OK、SFT 再実行したければ別 session で unsloth 環境作り直し。

In [ ]:
# [setup-install]  ←  このセルの名前です（以降のセルも1行目に名前を記載）
# 2026-10-02 問い合わせ対応版
# vLLM 0.25.1+cu129 wheel + peft + bitsandbytes + accelerate を uv で install
# uv が未 install なら先に pip install uv
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "show", "uv"], capture_output=True, text=True)
if r.returncode != 0:
    print("[install] uv を先に install...")
    !pip install -q uv 2>&1 | tail -3

# GitHub cu129 wheel を uv で install (--extra-index-url で torch cu129 系も引く)
!uv pip install --system \
    "https://github.com/vllm-project/vllm/releases/download/v0.25.1/vllm-0.25.1+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" \
    --extra-index-url https://download.pytorch.org/whl/cu129 \
    peft bitsandbytes accelerate > /tmp/pip_install.log 2>&1
print("exit code:", _)
!tail -30 /tmp/pip_install.log

# peft の torchao 版本チェック hard error 回避 (Colab pre-installed torchao 0.10 は要件不満足)
# Base fp16 merge では torchao 不要
!uv pip uninstall torchao 2>&1 | tail -3

# torchcodec を外す (動画は扱わないため不要)。
# Colab の torchcodec は CUDA 12 向けビルドで libnvrtc.so.12 を要求するが、Colab の torch は
# CUDA 13 版なのでそれを読み込まない。結果 torchcodec の native ライブラリが OSError で
# 失敗し、vllm の video import のガードは OSError を捕まえないため vllm の import 全体が
# 落ちる (OSError: Could not load this library: .../libtorchcodec_image.so)。
# 外しておけばガードが ImportError を見て placeholder を入れるので問題なく進む。
!uv pip uninstall torchcodec 2>&1 | tail -3

**⚠ 上のセル [setup-install] の実行後、「ランタイム」→「セッションを再起動する」を実施してください。**

その後、下のセル [config] 以降を実行してください。

vllm の初期化には fresh Python session が必要 (torch/CUDA 系の import 順序リセット)。
restart 後に `config` セル以降を順に実行。

## Step 1: 設定変数

**変更してよいのは `config` セルの「1」〜「4」の範囲だけ**。同じセルの「ここから下」以降と、
ほかのセルは変更できない。

| 区分 | 内容 |
|---|---|
| 1 | notebook1_sft と揃える値 (`RUN_ID` / `PATH_CHOICE` / `ADAPTER_SOURCE` / `STAGE*_SLUG` / `HF_ORG` / `HF_REPO_PREFIX`) |
| 2 | 提出 JSON に載せる recipe の表記 (`RECIPE` / `RECIPE_DETAIL`) |
| 3 | 推論パラメータ (`TEMPERATURE` / `SAMPLING_SEED` / `MAX_TOKENS_BY_TASK` / `MAX_MODEL_LEN` / `GPU_MEM_UTIL` / `DTYPE` / `ENFORCE_EAGER` / `CHUNK_SIZE`) |
| 4 | 配布物とベースモデルの場所 (`BASE_MODEL` / `INFERENCE_CONFIG_PATH`) |

1 の値は notebook1_sft と**完全に一致**させる必要がある。

`INFERENCE_CONFIG_PATH` は運営配布の推論用 config の置き場所。chat template と prompt の
既定値、および chat template が満たすべき条件 (`chat_template_policy`) が入っている。

**chat template は変更してよい** (学習時と同じ形式に合わせる場合など)。ただし問題文の前後に
現れる「枠組み」に指示文・例示・誘導を入れることはできない。入れてよいのは特殊トークンと
役割名と空白のみで、全問で同じ枠組みである必要がある。推論後に枠組みを提出 JSON に記録し、
最後のセルの形式チェックで確認する。

問題文の差し込み方 (単一の user turn)、評価用データの範囲 (`eval-data` セルの `CUTOFF`)、
提出 JSON の形式は変更できない。


In [ ]:
# [config]
# =====================================================================
# 設定セル
#
# **参加者が変更してよいのは、このセルの「1」〜「4」の範囲だけ**。
# 「ここから下」以降とほかのセルは変更できない。
#
# 変更してよいもの
#   - 使用するモデルの指定 (notebook1_sft で学習した adapter の指定)
#   - vLLM のオプションとして指定可能な推論パラメータ
#   - chat template (推論用 config 側。枠組みの条件を満たす範囲)
# 変更できないもの
#   - 問題文の差し込み方 (単一の user turn)
#   - 評価用データの範囲 (eval-data セルの CUTOFF)
#   - 提出 JSON の形式
# =====================================================================

# ---------------------------------------------------------------------
# 1. notebook1_sft と揃える値
# ---------------------------------------------------------------------
# RUN_ID: 出力ファイルの命名と、途中再開の判定に使う識別子。
# **notebook1_sft と必ず同じ値にすること**。日付から自動生成すると日をまたいだときに
# ずれるため、自分で決めた固定の文字列を書く。
RUN_ID = 't4_20261013_run02_prmfull'

# 実行 path (notebook1_sft の PATH_CHOICE と対応、提出 JSON の path フィールドになる)
#   "ii"  = notebook1 で学習した adapter を使う
#   "iii" = 学習なし、Base のまま推論する (adapter 不要)
PATH_CHOICE = "ii"              # "ii" | "iii"

ADAPTER_SOURCE = "drive"        # "drive" | "hf_hub"

# adapter の保存先ディレクトリ名 (notebook1_sft と揃える)
STAGE1_SLUG = 't4_20261013_run01_stage1'
STAGE2_SLUG = 't4_20261013_run02_prmfull_stage2'

# HF Hub のアカウント名。"auto" のままにすると、設定した token から自動で取得する。
# organization の下に置く場合など、明示したいときはアカウント名を書く
HF_ORG = "auto"
# HF Hub repo 名の接頭辞。notebook1_sft と同じ値にすること
HF_REPO_PREFIX = "geniac_prize_2026_"

# ---------------------------------------------------------------------
# 2. 提出 JSON に載せる recipe の表記
# ---------------------------------------------------------------------
#   RECIPE        : 配布時の既定の構成をそのまま使ったなら "default"、
#                   データセット・件数・ハイパーパラメータのいずれかを変えたなら "custom"
#   RECIPE_DETAIL : 具体的な構成 (自由記述)。独自の構成の場合はここに書く
#   ルート (iii) は学習していないため "custom" を指定する
RECIPE = 'custom'
RECIPE_DETAIL = 'Stage1: run01 Tulu persona IF 2000 reused; Stage2: PRM800K 1500 + GSM8K 500 full solutions, 1 epoch, lr 5e-5, LoRA r8 alpha16, maxlen2048, all nonpadding loss; heldout excluded'

# ---------------------------------------------------------------------
# 3. 推論パラメータ (変更可)
# ---------------------------------------------------------------------
# サンプリング設定。提出 JSON に記録され、監査での再現確認に使う
TEMPERATURE = 0.0                # 0.0 = greedy。再現性のため既定は 0.0
SAMPLING_SEED = 42               # temperature > 0 のときの再現用 seed

# 問題の種類ごとの出力長上限。上限を超える分は生成されない。
# 長い問題文では MAX_MODEL_LEN に収まるよう自動で切り詰められる
MAX_TOKENS_BY_TASK = {
    "AMPS_Hard": 2048,
    "math_comp": 2048,
    "olympiad": 4096,            # 証明を書く問題があるため長めに取る
}
MAX_TOKENS_DEFAULT = 2048        # 上の表に無い種類に使う値

# vLLM 構築時のオプション
MAX_MODEL_LEN = 4096             # 大きくすると KV cache が増え、同時処理数が減って遅くなる
GPU_MEM_UTIL = 0.85              # T4 のメモリのうち vLLM に割り当てる割合
DTYPE = "float16"                # T4 (compute 7.5) は bfloat16 非対応
ENFORCE_EAGER = False            # False で CUDA graph 有効 (速い)。初期化に失敗する場合は True

# 推論の分割単位。この件数ずつ処理し、各回のあとに Drive へ途中保存するため、
# セッションが切れても続きから再開できる。小さくすると保存が増えて安全だが遅くなる
CHUNK_SIZE = 58

# ---------------------------------------------------------------------
# 4. 配布物とベースモデルの場所
# ---------------------------------------------------------------------
# Base モデル (ルールで固定)
BASE_MODEL = "Qwen/Qwen3-4B-Base"

# 推論用 config ファイル (運営配布)。chat template の正本はこのファイル。
# 配布物の inference_config.json を Colab の /content/ にアップロードしておく。
INFERENCE_CONFIG_PATH = "/content/inference_config.json"

# =====================================================================
# ここから下は派生値と検査。**編集不要**。
# =====================================================================

# Drive workspace layout (ADAPTER_SOURCE="drive" 時の adapter 参照先)
WORKSPACE = "/content/drive/MyDrive/geniac_workspace"
DRIVE_STAGE1_DIR = WORKSPACE + "/adapters/" + STAGE1_SLUG
DRIVE_STAGE2_DIR = WORKSPACE + "/adapters/" + STAGE2_SLUG
PREDICTIONS_DIR = WORKSPACE + "/predictions"

# HF Hub の参照先 (ADAPTER_SOURCE="hf_hub" 時) は、HF_ORG が確定したあとに
# hf-login セルで組み立てる

# Colab local disk (transient)
LOCAL_WORK = "/content/sft_work"
STAGE1_ADAPTER_LOCAL = LOCAL_WORK + "/stage1_adapter"    # HF Hub pull 時の一時保存
STAGE2_ADAPTER_LOCAL = LOCAL_WORK + "/stage2_adapter"
MERGED_FINAL_LOCAL = LOCAL_WORK + "/merged_final"        # Base + Stage1 + Stage2 merged (vLLM load 元)

# 推論用 config を読み込む (chat template と prompt の組み立て方の正本)。
# ここで読んだ値をそのまま使うため、notebook 側で template を書き換える必要はない。
import hashlib as _hashlib
import json as _json
import os as _os

if not _os.path.exists(INFERENCE_CONFIG_PATH):
    raise FileNotFoundError(
        "推論用 config が見つかりません: " + INFERENCE_CONFIG_PATH +
        "。配布物の inference_config.json を Colab の /content/ にアップロードしてください"
    )
with open(INFERENCE_CONFIG_PATH, encoding="utf-8") as _f:
    INFERENCE_CONFIG = _json.load(_f)

CHAT_TEMPLATE = INFERENCE_CONFIG["chat_template"]["jinja"]
CHAT_TEMPLATE_SHA256 = _hashlib.sha256(CHAT_TEMPLATE.encode("utf-8")).hexdigest()
CHAT_TEMPLATE_IS_DEFAULT = (
    CHAT_TEMPLATE_SHA256 == INFERENCE_CONFIG["chat_template"]["sha256"]
)
# chat template は変更してよい (学習時と同じ形式に合わせる場合など)。ただし問題文の
# 前後に現れる「枠組み」に指示文や例示を入れることはできない。条件は config の
# chat_template_policy に記載しており、推論後に自動で確認する。
FRAMING_POLICY = INFERENCE_CONFIG["chat_template_policy"]
if INFERENCE_CONFIG["base_model"] != BASE_MODEL:
    raise ValueError(
        "BASE_MODEL が config の指定と異なります (config: " +
        INFERENCE_CONFIG["base_model"] + " / 設定値: " + BASE_MODEL + ")"
    )

assert PATH_CHOICE in ("ii", "iii"), "PATH_CHOICE は \"ii\" か \"iii\""
assert RECIPE in ("default", "custom"), "RECIPE は default か custom"
assert CHUNK_SIZE >= 1, "CHUNK_SIZE は 1 以上"

# 推論に使うモデル: ルート (iii) は Base をそのまま、それ以外は merge 済モデル
INFERENCE_MODEL = BASE_MODEL if PATH_CHOICE == "iii" else MERGED_FINAL_LOCAL
USE_CHAT_TEMPLATE = PATH_CHOICE != "iii"   # Base は chat_template を持たないため raw prompt

print("[config] 推論用 config: " + INFERENCE_CONFIG_PATH +
      " (version " + INFERENCE_CONFIG["config_version"] + ")")
print("[config] chat template: sha256 " + CHAT_TEMPLATE_SHA256[:16] +
      (" (配布時の既定値)" if CHAT_TEMPLATE_IS_DEFAULT else " (既定値から変更されています)"))
print()
print("--- notebook1_sft と揃える値 ---")
print("  RUN_ID          :", RUN_ID)
print("  PATH_CHOICE     :", PATH_CHOICE)
print("  ADAPTER_SOURCE  :", ADAPTER_SOURCE)
print("  STAGE1_SLUG     :", STAGE1_SLUG)
print("  STAGE2_SLUG     :", STAGE2_SLUG)
print("--- 推論パラメータ ---")
print("  TEMPERATURE     :", TEMPERATURE, "/ seed", SAMPLING_SEED)
print("  MAX_TOKENS      :", MAX_TOKENS_BY_TASK, "/ default", MAX_TOKENS_DEFAULT)
print("  MAX_MODEL_LEN   :", MAX_MODEL_LEN, "/ gpu_mem", GPU_MEM_UTIL,
      "/ dtype", DTYPE, "/ enforce_eager", ENFORCE_EAGER)
print("  CHUNK_SIZE      :", CHUNK_SIZE)
print("--- 推論対象 ---")
print("  BASE_MODEL      :", BASE_MODEL)
print("  INFERENCE_MODEL :", INFERENCE_MODEL,
      "(chat_template:", "ON)" if USE_CHAT_TEMPLATE else "OFF)")


In [ ]:
# [hf-login]
# HF Hub 認証。**token の設定は必須**。
# 評価用データは利用条件への同意が必要なため、token がないと取得できない。
# あわせて Base モデルのダウンロードも速くなる。
import os
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get("HFH_READ")
except Exception as e:
    raise RuntimeError(
        "Hugging Face の read token が設定されていません。次の手順で設定してください。\n"
        "  1. https://huggingface.co/settings/tokens で read 権限の token を作る\n"
        "  2. Colab 左サイドバーの鍵アイコン (Secrets) → 新規追加 →\n"
        "     Name=HFH_READ, Value=作った token → ノートブックからのアクセスを有効にする\n"
        "  3. 評価用データのページで、ログインした状態で利用条件に同意する (自動承認)"
    ) from e

login(token=hf_token, add_to_git_credential=False)
os.environ["HF_TOKEN"] = hf_token
os.environ["HF_HUB_DISABLE_XET"] = "1"
print("[HF login] OK")

# HF_ORG が "auto" のときは、設定した token からアカウント名を取得する
if HF_ORG == "auto":
    try:
        from huggingface_hub import whoami
        HF_ORG = whoami()["name"]
        print("[HF] HF_ORG を token から取得しました: " + HF_ORG)
    except Exception as e:
        print("[HF] HF_ORG を自動取得できませんでした (" + type(e).__name__ + ")")
        print("  Hugging Face Hub を使う場合は、config セルの HF_ORG に")
        print("  自分のアカウント名を書いてから実行し直してください")

# adapter を HF Hub から取得する場合の参照先 (Drive 側と対称に、ここで明示する)
HF_STAGE1_REPO = HF_ORG + "/" + HF_REPO_PREFIX + STAGE1_SLUG
HF_STAGE2_REPO = HF_ORG + "/" + HF_REPO_PREFIX + STAGE2_SLUG
if ADAPTER_SOURCE == "hf_hub":
    assert HF_ORG and HF_ORG != "auto", (
        "HF_ORG が決まっていません。config セルに自分のアカウント名を書いてください")
    print("  HF_STAGE1_REPO:", HF_STAGE1_REPO)
    print("  HF_STAGE2_REPO:", HF_STAGE2_REPO)

In [ ]:
# [adapter-prepare]
# Drive のマウントと出力先ディレクトリの作成は**全ルート共通**。
# 推論結果と途中保存は Drive に書くため、学習しないルート (iii) でも必要になる。
# ルートと ADAPTER_SOURCE によって変わるのは adapter の準備だけ。
import os
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
os.makedirs(LOCAL_WORK, exist_ok=True)
print("PREDICTIONS_DIR:", PREDICTIONS_DIR)
print("LOCAL_WORK:", LOCAL_WORK)

if PATH_CHOICE == "iii":
    print("[path iii] adapter は不要のため skip (Base をそのまま推論)")
else:
    if ADAPTER_SOURCE == "drive":
        STAGE1_ADAPTER_DIR = DRIVE_STAGE1_DIR
        STAGE2_ADAPTER_DIR = DRIVE_STAGE2_DIR
        for d in [STAGE1_ADAPTER_DIR, STAGE2_ADAPTER_DIR]:
            assert os.path.isdir(d), "adapter dir not found: " + d + " (notebook1_sft を先に実行してください)"
            print("  " + d + " OK (" + str(len(os.listdir(d))) + " files)")

    elif ADAPTER_SOURCE == "hf_hub":
        # 上流の HF login cell で hf_token 設定済 (必須)
        assert os.environ.get("HF_TOKEN"), "HFH_READ が Colab Secrets に未設定、HF Hub からの adapter pull 不可"
        from huggingface_hub import snapshot_download
        hf_token = os.environ["HF_TOKEN"]

        os.makedirs(STAGE1_ADAPTER_LOCAL, exist_ok=True)
        os.makedirs(STAGE2_ADAPTER_LOCAL, exist_ok=True)
        for repo_id, local_dir in [(HF_STAGE1_REPO, STAGE1_ADAPTER_LOCAL),
                                   (HF_STAGE2_REPO, STAGE2_ADAPTER_LOCAL)]:
            print("[HF Hub pull] " + repo_id + " -> " + local_dir)
            snapshot_download(repo_id=repo_id, local_dir=local_dir, token=hf_token)
        STAGE1_ADAPTER_DIR = STAGE1_ADAPTER_LOCAL
        STAGE2_ADAPTER_DIR = STAGE2_ADAPTER_LOCAL

    else:
        raise ValueError("ADAPTER_SOURCE must be 'drive' or 'hf_hub', got: " + ADAPTER_SOURCE)

    print("STAGE1_ADAPTER_DIR:", STAGE1_ADAPTER_DIR)
    print("STAGE2_ADAPTER_DIR:", STAGE2_ADAPTER_DIR)
    print("MERGED_FINAL_LOCAL:", MERGED_FINAL_LOCAL)

## Step 2: Base fp16 load → adapter chain apply → merged 保存

**flow**:
1. Base (`Qwen/Qwen3-4B-Base`) を fp16 で GPU load (~8GB VRAM、~2 min)
2. Stage 1 adapter (`unsloth/qwen3-4b-base-unsloth-bnb-4bit` 由来) apply → `merge_and_unload()` (in-memory、~0.3 min)
3. Stage 2 adapter (Stage 1 merged 由来) apply → `merge_and_unload()` (in-memory、~0.2 min)
4. 完成 merged を Colab local disk `/content/sft_work/merged_final` に fp16 保存 (~8GB、**~11 min**)
5. tokenizer は Stage 2 adapter dir から load (chat_template 継承)

**VRAM 予測** (T4 15GB):
- Base fp16 load: ~8GB
- adapter merge (2 回連鎖): peak ~8.2GB (adapter 一時 130MB + base)
- merged 保存: disk I/O only、VRAM 圧迫なし

**Stage A の所要時間の目安**: 約 15 分

In [ ]:
# [base-load]
if PATH_CHOICE == "iii":
    print("[path iii] Base の事前 load は不要 (vLLM が直接 load するため skip)")
else:
    # Base fp16 load (~8GB VRAM、~2 min)
    import time
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    t0 = time.time()
    print("[Base load] " + BASE_MODEL + " fp16 → GPU")
    # 注: transformers 5.x では torch_dtype → dtype に rename (旧引数は deprecation warning のみ、動作 OK)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        dtype=torch.float16,
        device_map="cuda",
        low_cpu_mem_usage=True,
    )
    print("[Base load] done in %.1f min" % ((time.time() - t0) / 60))
    print("  VRAM: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

In [ ]:
# [stage1-merge]
if PATH_CHOICE == "iii":
    print("[path iii] Stage 1 adapter の適用なし")
else:
    # Stage 1 adapter apply + merge_and_unload
    import time, torch
    from peft import PeftModel

    t0 = time.time()
    print("[Stage 1 apply] " + STAGE1_ADAPTER_DIR)
    model = PeftModel.from_pretrained(model, STAGE1_ADAPTER_DIR)
    print("  after apply VRAM: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

    print("[Stage 1 merge] merge_and_unload...")
    model = model.merge_and_unload()
    torch.cuda.empty_cache()
    print("[Stage 1 apply + merge] done in %.1f min" % ((time.time() - t0) / 60))
    print("  after merge VRAM: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

In [ ]:
# [stage2-merge]
if PATH_CHOICE == "iii":
    print("[path iii] Stage 2 adapter の適用なし")
else:
    # Stage 2 adapter apply + merge_and_unload
    import time, torch
    from peft import PeftModel

    t0 = time.time()
    print("[Stage 2 apply] " + STAGE2_ADAPTER_DIR)
    model = PeftModel.from_pretrained(model, STAGE2_ADAPTER_DIR)
    print("  after apply VRAM: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

    print("[Stage 2 merge] merge_and_unload...")
    model = model.merge_and_unload()
    torch.cuda.empty_cache()
    print("[Stage 2 apply + merge] done in %.1f min" % ((time.time() - t0) / 60))
    print("  after merge VRAM: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

In [ ]:
# [merged-save]
if PATH_CHOICE == "iii":
    print("[path iii] merge 済モデルの保存は不要 (skip)")
else:
    # merged fp16 を local disk に save (~8GB、Colab local disk 遅くて ~11 min)
    # tokenizer は Stage 2 adapter dir から load (chat_template 継承)
    import os, time
    from transformers import AutoTokenizer

    os.makedirs(MERGED_FINAL_LOCAL, exist_ok=True)

    t0 = time.time()
    print("[save] merged → " + MERGED_FINAL_LOCAL + " (~11 min 想定、Colab local disk 遅い)")
    model.save_pretrained(MERGED_FINAL_LOCAL, safe_serialization=True)
    print("[save] merged done in %.1f min" % ((time.time() - t0) / 60))

    tokenizer = AutoTokenizer.from_pretrained(STAGE2_ADAPTER_DIR)
    # 推論に使う chat_template は config のものに揃える。adapter 側に別の template が
    # 入っていた場合は、どちらを使ったかがはっきりするようここで知らせる。
    if getattr(tokenizer, "chat_template", None) != CHAT_TEMPLATE:
        print("[save] adapter の chat_template と config が異なります。config の値を使います")
    tokenizer.chat_template = CHAT_TEMPLATE
    tokenizer.save_pretrained(MERGED_FINAL_LOCAL)
    print("[save] tokenizer done (chat_template = 推論用 config の値)")

    files = sorted(os.listdir(MERGED_FINAL_LOCAL))
    total_gb = sum(os.path.getsize(MERGED_FINAL_LOCAL + "/" + f) for f in files) / 1e9
    print("  " + MERGED_FINAL_LOCAL + " total: %.2f GB" % total_gb)
    for f in files:
        size_mb = os.path.getsize(MERGED_FINAL_LOCAL + "/" + f) / 1e6
        print("    " + f + "  (%.1f MB)" % size_mb)

## Step 3: **⚠ Runtime restart 必須** (VRAM 完全解放のため)

**`PATH_CHOICE="iii"` の場合は restart 不要**。`base-load` 〜 `merged-save` を実行していないため
VRAM を消費していない。そのまま Step 4 へ進んでよい。

### なぜ restart が必要か

`base-load` 〜 `stage2-merge` の Base load + adapter merge で **VRAM 8GB を消費**している。この段階で
`del model + gc.collect() + torch.cuda.empty_cache()` を試みても、Jupyter の内部参照や PyTorch の
CUDA context により **VRAM は解放されない**。

そのまま `llm-build` を実行すると:
- vLLM が `gpu_memory_utilization=0.85` で 13.3GB 割当を試行
- 既に 8.1GB 使用中なので空き 7.5GB のみ
- Model load + KV cache 割当で **OOM crash** (`Engine core initialization failed`)

### 手順

1. **「ランタイム」 → 「セッションを再起動する」** (メニューから実行)
2. Restart 完了後、以下の名前のセルを順に実行:
   - `[config]` ← 再実行必要
   - `[hf-login]` ← 再実行必要
   - `[adapter-prepare]` ← 再実行必要
   - `[eval-data]`
   - `[llm-build]`
   - `[inference]`
   - `[predictions-save]`
3. **`base-load` 〜 `merged-save` は再実行不要** (merged model は `/content/sft_work/merged_final` に保存済)

**Colab local disk は restart で消えない**ため、merged fp16 8GB は保持される。


## Step 4: 評価用データの準備

採点対象の問題を読み込む。取得と絞り込みは `math_sft.evalset` が行い、**採点に関わる列
(正解など) はその中で落としている**ため、手元に正解は残らない。ルール上、評価用データの
内容を参照することは禁止されている。

対象範囲は変更できない。


In [ ]:
# [eval-data]
# 評価用データの準備。取得と絞り込みは math_sft.evalset が行う。
# 採点に関わる列 (正解など) はその中で落としているため、手元に正解は残らない。
# 対象範囲は変更できない (参加者が変更してよい設定ではないため config セルには置かない)。
import os, sys, tarfile

if not os.path.isdir("/content/math_sft"):
    assert os.path.exists("/content/math_sft.tar.gz"), (
        "math_sft.tar.gz が /content/ にありません。配布物をアップロードしてください")
    with tarfile.open("/content/math_sft.tar.gz") as tf:
        tf.extractall("/content")
    print("[setup] math_sft を展開しました")
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

from math_sft.evalset import load_eval_questions

questions = load_eval_questions()
print("[eval] 対象問題数:", len(questions))

## Step 5: vLLM 構築 (Runtime restart 後、fresh state で実行)

**前提**: Step 3 の restart 済 → VRAM = 0GB クリーンな状態。

**vllm 0.25.1 Colab 対応 patch**:
`sys.stdout.fileno()` を Colab の ipykernel が非対応のため、vllm の `suppress_stdout()` を
`contextlib.nullcontext` に置換する必要あり。この monkey-patch を LLM 構築の**前**に必ず実行。

**T4 向けの設定** (いずれも `config` セルで変更できる):
- `dtype="float16"` — T4 (compute 7.5) は bfloat16 に対応していない
- `enforce_eager=False` — CUDA graph が有効になり速い。初期化に失敗する場合は `True` にする
- `max_model_len=4096` — 大きくすると KV cache が増え、同時処理数が減って遅くなる
- `gpu_memory_utilization=0.85` — T4 のメモリのうち vLLM に割り当てる割合
- `MAX_TOKENS_BY_TASK` — 問題の種類ごとの出力長上限。証明を書く種類だけ長めに取っている

**load 時間の目安**: 約 1.6 分 (restart 前に実行すると失敗する)

In [ ]:
# [llm-build]
# 2026-10-02 問い合わせ対応版
# vLLM 構築 (fresh state 前提、restart 済ませておくこと)
import gc, torch, time
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print("[gc] pre-check VRAM: %.1f GB (期待: <1 GB, restart 後は 0.0GB のはず)" %
      (torch.cuda.memory_allocated() / 1e9))

# vllm 0.25.1 は CUDA 12.9 向けのビルドで libcudart.so.12 などを必要とする。Colab の torch は
# CUDA 13 版 (2.11.0+cu130) なので、torch は CUDA 13 のライブラリしか読み込まない。
# 必要な cu12 ライブラリ自体は vllm の依存 (nvidia-*-cu12) で入っているため、場所を教える。
#
# 2 段構えにしているのは、vLLM が別プロセスを起こすため:
#   - LD_LIBRARY_PATH … モデル検査やエンジン worker の子プロセス向け (環境変数は継承され、
#     子プロセスの起動時にローダが読む)。これが無いと
#     「Model architectures ['Qwen3ForCausalLM'] failed to be inspected」で止まる
#   - ctypes の先読み … このプロセス向け (起動後の LD_LIBRARY_PATH 変更は自分には効かない)
import ctypes, glob, os

_cu12 = sorted({os.path.dirname(p) for p in glob.glob(
    "/usr/local/lib/python3*/dist-packages/nvidia/**/lib*.so.12", recursive=True)})
if _cu12:
    os.environ["LD_LIBRARY_PATH"] = ":".join(
        _cu12 + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(":")
    print("[patch] LD_LIBRARY_PATH に CUDA 12 を追加 (" + str(len(_cu12)) + " 箇所)")
    for _d in _cu12:
        print("    " + _d)
else:
    print("[patch] CUDA 12 のライブラリが見つかりません。setup-install セルを実行済みか確認してください")

_cudart = glob.glob(
    "/usr/local/lib/python3*/dist-packages/nvidia/cuda_runtime/lib/libcudart.so.12")
if _cudart:
    ctypes.CDLL(_cudart[0], mode=ctypes.RTLD_GLOBAL)
    print("[patch] libcudart.so.12 を先に読み込みました -> " + _cudart[0])

# vllm 0.25.1 Colab 非互換 patch (import 直後に monkey-patch)
# Colab の ipykernel は sys.stdout.fileno() に対応しないため、vllm の suppress_stdout を
# 無効化する。vllm/__init__.py は遅延 import 用の __getattr__ を持っており、
# 「import vllm.utils.system_utils」が成功しても vllm.utils 属性は設定されない
# (AttributeError: module vllm has no attribute utils)。モジュールを名前で受け取る。
import contextlib
import importlib

_vllm_sysutils = importlib.import_module("vllm.utils.system_utils")
if hasattr(_vllm_sysutils, "suppress_stdout"):
    _vllm_sysutils.suppress_stdout = contextlib.nullcontext
    print("[patch] suppress_stdout → contextlib.nullcontext")
else:
    print("[patch] suppress_stdout が無いため patch は不要です")

# LLM 構築 (ルート (iii) は Base、それ以外は merge 済モデル)
print("[LLM] model = " + INFERENCE_MODEL)
from vllm import LLM

t0 = time.time()
llm = LLM(
    model=INFERENCE_MODEL,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEM_UTIL,
    dtype=DTYPE,
    enforce_eager=ENFORCE_EAGER,
    trust_remote_code=True,
)
print("[LLM] load done in %.1f min" % ((time.time() - t0) / 60))

# vLLM 側の tokenizer に入っている chat_template が推論用 config と一致するか確認
if USE_CHAT_TEMPLATE:
    try:
        _vllm_ct = getattr(llm.get_tokenizer(), "chat_template", None)
        if _vllm_ct == CHAT_TEMPLATE:
            print("[LLM] chat_template: 推論用 config と一致")
        else:
            print("[LLM] 警告: vLLM 側の chat_template が推論用 config と一致しません")
    except Exception as e:
        print("[LLM] chat_template の確認をスキップしました (" + type(e).__name__ + ")")


## Step 6: 推論 (chunked、session 死耐性)

**分割実行**: `CHUNK_SIZE` 問ずつに分けて実行し、各回の完了後に Drive へ途中保存する。
セッションが切れても、そこまでの結果は Drive に残る。

**再開**: 同じセルを再実行すると Drive の途中結果から自動で続きを実行する。

**T4 での所要時間の目安**: 約 60 分。vLLM の continuous batching により最初の数問は
warm-up で遅く、中盤から 90〜110 tok/s 程度に上がる。

In [ ]:
# [inference]
# 推論 + predictions 構築 (chunked inference + periodic Drive save)
# 問題の種類ごとに出力長上限を変える (config セルの MAX_TOKENS_BY_TASK)。
# CHUNK_SIZE 問ずつに分割し、各回のあとに Drive へ保存してセッション切断に備える
import json, os, time
from vllm import SamplingParams

# prompt の組み立て方は推論用 config の指定どおりに行う (system prompt なし、問題文を
# 単一の user turn として渡す)。ここは参加者が変更してはならない。
_prompt_spec = INFERENCE_CONFIG["prompt"]
_placeholder = _prompt_spec["placeholder"]


def _build_messages(question_text):
    """推論用 config の turns 定義に問題文を差し込んで messages を組む。"""
    msgs = []
    if _prompt_spec["system_prompt"]:
        msgs.append({"role": "system", "content": _prompt_spec["system_prompt"]})
    for turn in _prompt_spec["turns"]:
        msgs.append({
            "role": turn["role"],
            "content": turn["content"].replace(_placeholder, question_text),
        })
    return msgs


# 実際に適用される chat template の「枠組み」(問題文の前後に付く部分) を記録する。
# vLLM が使う tokenizer から取るので、記録と実際の推論が食い違うことはない。
# ここでの判定はしない。提出 JSON に残し、形式チェックと監査で確認する。
CHAT_TEMPLATE_RECORD = {"applied": bool(USE_CHAT_TEMPLATE)}
if USE_CHAT_TEMPLATE:
    _tok = llm.get_tokenizer()
    _framings = set()
    for _q in questions:
        _text = _q["turns"][0]
        _rendered = _tok.apply_chat_template(
            _build_messages(_text), tokenize=False, add_generation_prompt=True)
        if _text in _rendered:
            _cut = _rendered.index(_text)
            _framings.add((_rendered[:_cut], _rendered[_cut + len(_text):]))
        else:
            _framings.add((None, None))
    _prefix, _suffix = sorted(_framings)[0]
    CHAT_TEMPLATE_RECORD.update({
        "sha256": _hashlib.sha256(
            (getattr(_tok, "chat_template", None) or "").encode("utf-8")).hexdigest(),
        "is_default": CHAT_TEMPLATE_IS_DEFAULT,
        "framing_variants": len(_framings),
        "framing_prefix": _prefix,
        "framing_suffix": _suffix,
    })
    print("[chat template] 枠組み " + str(len(_framings)) + " 種類 / " +
          str(len(_prefix or "") + len(_suffix or "")) + " 文字")
    print("  prefix: " + repr(_prefix))
    print("  suffix: " + repr(_suffix))
else:
    CHAT_TEMPLATE_RECORD.update({"framing_prefix": "", "framing_suffix": ""})
    print("[chat template] 適用なし (ルート iii は問題文をそのまま入力)")


# ルート (ii) は chat_template 適用 (llm.chat)、ルート (iii) は raw prompt (llm.generate)
# Base モデルは chat_template を持たないため、(iii) で chat を使うと出力が壊れる
# (config の note にも同じ扱いを記載している)
if USE_CHAT_TEMPLATE:
    conversations = [_build_messages(q["turns"][0]) for q in questions]
else:
    conversations = [q["turns"][0] for q in questions]

# task 別 SamplingParams (per-request、vLLM は list 受付 OK)
sampling_params_list = [
    SamplingParams(
        temperature=TEMPERATURE,
        seed=SAMPLING_SEED,
        max_tokens=MAX_TOKENS_BY_TASK.get(q["task"], MAX_TOKENS_DEFAULT),
    )
    for q in questions
]

# 内訳表示 (debug 用)
from collections import Counter
task_max_tokens = Counter(
    (q["task"], MAX_TOKENS_BY_TASK.get(q["task"], MAX_TOKENS_DEFAULT))
    for q in questions
)
print("[sampling] task 別 max_tokens:")
for (task, mt), cnt in sorted(task_max_tokens.items()):
    print("  " + task + ": max_tokens=" + str(mt) + " (n=" + str(cnt) + ")")

# chunked inference (CHUNK_SIZE 問ずつ、各 chunk 後に Drive save)
# CHUNK_SIZE は config セルで設定する
predictions_all = []
partial_drive_path = PREDICTIONS_DIR + "/" + RUN_ID + "_partial.json"
partial_local_path = LOCAL_WORK + "/" + RUN_ID + "_partial.json"

# 途中経過がどの構成のものかを表す情報。再開時にこれが一致しない場合は別物とみなす。
# RUN_ID だけで判定すると、RUN_ID を変え忘れたまま別 recipe を実行したときに
# 前回の予測と混ざるため (#3)。
run_signature = {
    "run_id": RUN_ID,
    "path_choice": PATH_CHOICE,
    "stage1_slug": STAGE1_SLUG,
    "stage2_slug": STAGE2_SLUG,
    "adapter_source": ADAPTER_SOURCE,
    "base_model": BASE_MODEL,
}


def _load_partial(path, label):
    """途中経過を読み込む。構成が一致しない場合は使わない。"""
    existing = json.load(open(path))
    saved = existing.get("run_signature")
    if saved is None:
        print("[skip resume] " + label + ": 構成情報のない古い途中経過のため使用しません")
        return None
    diff = [k for k, v in run_signature.items() if saved.get(k) != v]
    if diff:
        print("[skip resume] " + label + ": 構成が一致しないため使用しません (相違: " + ", ".join(diff) + ")")
        for k in diff:
            print("    " + k + ": 途中経過=" + str(saved.get(k)) + " / 今回=" + str(run_signature[k]))
        print("    別の構成で最初から実行します。前回の結果を残したい場合は RUN_ID を変えてください。")
        return None
    preds = existing.get("predictions", [])
    print("[resume] " + label + ": " + str(len(preds)) + " predictions を読み込み")
    return preds


# 既存 partial 再開判定 (Drive 優先 = session 跨ぎ対応、local fallback = 同一 session)
if os.path.exists(partial_drive_path):
    predictions_all = _load_partial(partial_drive_path, "Drive (session 跨ぎ)") or []
elif os.path.exists(partial_local_path):
    predictions_all = _load_partial(partial_local_path, "local (同一 session)") or []
else:
    print("[start] 新規実行、chunk_size=" + str(CHUNK_SIZE))

# resume 済み分を skip して残りを処理
n_done = len(predictions_all)
print("[inference] start: " + str(len(conversations) - n_done) + " conversations (残)")

t_total = time.time()
for i in range(n_done, len(conversations), CHUNK_SIZE):
    chunk_convs = conversations[i:i+CHUNK_SIZE]
    chunk_params = sampling_params_list[i:i+CHUNK_SIZE]
    chunk_qs = questions[i:i+CHUNK_SIZE]

    print("[chunk " + str(i//CHUNK_SIZE + 1) + "] " + str(i) + "-" + str(i+len(chunk_convs)-1) + " (" + str(len(chunk_convs)) + " prompts)")
    t0 = time.time()
    if USE_CHAT_TEMPLATE:
        outputs = llm.chat(chunk_convs, chunk_params, use_tqdm=True)
    else:
        outputs = llm.generate(chunk_convs, chunk_params, use_tqdm=True)
    elapsed_chunk = time.time() - t0
    print("[chunk " + str(i//CHUNK_SIZE + 1) + "] " + str(len(outputs)) + " done in %.1f min" % (elapsed_chunk / 60))

    # predictions 追加
    for q, out in zip(chunk_qs, outputs):
        predictions_all.append({
            "question_id": q["question_id"],
            "task": q["task"],
            "subtask": q.get("subtask", ""),
            "answer_text": out.outputs[0].text,
            "prompt_tokens": len(out.prompt_token_ids),
            "completion_tokens": len(out.outputs[0].token_ids),
        })

    # 段階保存 (Drive + local 両方、session 死時の recovery 用)
    partial_data = {"predictions": predictions_all, "n_done": len(predictions_all),
                    "run_signature": run_signature}
    with open(partial_drive_path, "w") as f:
        json.dump(partial_data, f, ensure_ascii=False)
    with open(partial_local_path, "w") as f:
        json.dump(partial_data, f, ensure_ascii=False)
    print("[chunk " + str(i//CHUNK_SIZE + 1) + "] partial saved: " +
              str(len(predictions_all)) + "/" + str(len(questions)))

elapsed = time.time() - t_total
print()
print("[inference] TOTAL " + str(len(predictions_all)) + " in %.1f min" % (elapsed / 60) +
      " (%.1f s/problem avg)" % (elapsed / max(1, len(predictions_all) - n_done)) if len(predictions_all) > n_done else "[inference] TOTAL " + str(len(predictions_all)) + " (all from cache)")

predictions = predictions_all
print("[predictions] " + str(len(predictions)))
if predictions:
    print("[preview] first answer (200 chars): " + repr(predictions[0]["answer_text"][:200]))

    # 統計
    avg_comp = sum(p["completion_tokens"] for p in predictions) / len(predictions)
    avg_prompt = sum(p["prompt_tokens"] for p in predictions) / len(predictions)
    print("[stats] avg prompt tokens: %.0f" % avg_prompt + ", avg completion tokens: %.0f" % avg_comp)

## Step 7: predictions.json を Drive に保存

Omnicampusが読む形式で保存。`submission_meta` に recipe / RUN_ID / colab_env を含める。

In [ ]:
# [predictions-save]
# predictions.json 出力 (Drive)
import json, os
import datetime as _dt

submission = {
    "submission_meta": {
        # --- スキーマ必須フィールド ---
        "path": PATH_CHOICE,
        "base_model": BASE_MODEL,
        "recipe": RECIPE,
        # --- 以下は補足情報 (スキーマ外、再現性と不正調査のため付与) ---
        "recipe_detail": RECIPE_DETAIL,
        "run_id": RUN_ID,
        "stage1_slug": STAGE1_SLUG,
        "stage2_slug": STAGE2_SLUG,
        "adapter_source": ADAPTER_SOURCE,
        "generated_at": _dt.datetime.now(_dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "colab_env": "t4",
        "vllm_version": "0.25.1+cu129",
        # 推論に実際に適用した chat template の記録 (形式チェックと監査で確認する)
        "chat_template": CHAT_TEMPLATE_RECORD,
        "inference_config": {
            "max_model_len": MAX_MODEL_LEN,
            "max_tokens_by_task": MAX_TOKENS_BY_TASK,
            "dtype": DTYPE,
            "enforce_eager": ENFORCE_EAGER,
            "gpu_memory_utilization": GPU_MEM_UTIL,
            "temperature": TEMPERATURE,
            "seed": SAMPLING_SEED,
        },
    },
    "predictions": predictions,
}

drive_output = PREDICTIONS_DIR + "/" + RUN_ID + ".json"
with open(drive_output, "w") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)

local_output = LOCAL_WORK + "/" + RUN_ID + ".json"
with open(local_output, "w") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)

print("[predictions] saved:")
print("  Drive: " + drive_output + " (%.2f MB)" % (os.path.getsize(drive_output) / 1e6))
print("  Local: " + local_output + " (%.2f MB)" % (os.path.getsize(local_output) / 1e6))

In [ ]:
# [check-submission]
# 提出ファイルが採点可能な形式かを確認する (正誤やスコアは判定しません)。
# 配布している check_submission.py と同じ内容です。

#!/usr/bin/env python3
"""提出ファイルの形式チェック (配布している check_submission.py と同じ内容)。

Distributed to participants in place of the grading notebook. It deliberately reports
nothing about correctness: no score, no per-question result, no ground truth. Its only
job is to catch a file that the grader would reject or mis-handle.

The embedded QUESTION_IDS are the opaque identifiers of the evaluation questions. They
contain no question text and no answers.

Usage:
    python check_submission.py path/to/submission.json
"""
from __future__ import annotations

import json
import os
import re
import sys

TASKS = ("AMPS_Hard", "math_comp", "olympiad")
PATHS = ("ii", "iii")
RECIPES = ("default", "custom")
EXPECTED_BASE_MODEL = "Qwen/Qwen3-4B-Base"
REQUIRED_META = ("path", "base_model", "recipe", "generated_at", "colab_env")
REQUIRED_PREDICTION = {
    "question_id": str,
    "task": str,
    "subtask": str,
    "answer_text": str,
    "prompt_tokens": int,
    "completion_tokens": int,
}
# Fields that would mean the file carries grading results rather than raw predictions.
SCORE_FIELDS = ("score", "rescored_score", "metrics_summary", "correct", "is_correct")

# --- chat template framing policy -------------------------------------------------
# The "framing" is what the chat template renders around the question text. Structural
# changes are allowed (a participant may have trained with a different chat format), but
# the framing must not carry content: no instructions, no exemplars, no guidance. The
# check is an allowlist, so it cannot be worked around by reading this file.
MAX_FRAMING_CHARS = 128
ALLOWED_FRAMING_WORDS = frozenset("""
system user assistant tool human ai bot
question answer problem solution instruction response input output prompt
inst sys start end of text turn header
q a s e eos bos eot im
""".split())
REFERENCE_FRAMING = ("<|im_start|>user\n", "<|im_end|>\n<|im_start|>assistant\n")
_SPECIAL_TOKEN = re.compile(r"<\|[^|>]{0,64}\|>")
_FRAMING_WORD = re.compile(r"[A-Za-z][A-Za-z0-9_]*")


def framing_words(text: str) -> list:
    """Alphabetic runs left after removing <|...|> special tokens."""
    return _FRAMING_WORD.findall(_SPECIAL_TOKEN.sub(" ", text))

QUESTION_IDS = frozenset("""
ef6c6830ebf0e71d74e7ceb45a242f2b87228933eebfe47db8d798d5738cdaeb
6e12ab2219a4051c396de5afc4228be52c2d8617c7f5621be8c364f50016c80d
c03217423ccb2181ef9e3106b4c116aaf1bcdbd45b3a97e04110665690177eca
698d9bd9ad134096358dad82317dae610d9f265255aeee52c644871b83eb7a85
56155e5b8c1dc430cc262691aea0a75d71f7dee78b6e15304607caf0b56ecfbd
0d3a48049dc5da31531654f59e7866832579f812b892610bd1b178fc4d010d0c
3f55db366e970e1965bbba20e9958a4c0ea52e4224d11d875d390983485dd32b
3649fb05ed87c2f3ed8a933e7c8f032b54a5d6f0d23504cdd578541dd60c0f08
e5b72d9cdb39c6e5f8798a557f119fff35d98fd165f55659e71bf6ac38b5f178
bc3551b5f643d8920f428909104fe2640a50f6b8a17fff6196aa78f25df28bb6
2dd75e080f3e276f3dcf304d970e6a03973ddf777537422c9aba59559ddc9594
558c3c463860fcaa2b08edd9d5f0f11071944bf7d9229574862a92e6d91f1a21
e81b96eed50d6b9b43f041d6a36ad0d9a4d00038c42c47a16efd4465d79d08a4
2191e02b65ac78dd0d28bc13f41786d2596dbaa560ad9d0658a27ae2814faef3
5b9c56e9392dd999162474f10addf1a3a72f1d8b1c8bafc999ccec6f22194c18
fc903ff1f1f66a6096c9dea58c278b48f54c80a606b65f82af1730559d73755f
e56e4d8ed64cb6d0e43ce3c8edda0a77d769b4bda5230c1aba57442e27214694
749116060528890530a40d111c5506f4cdec004a7c52a87ef0a4a01206c187ec
6d9bd0806e27f34ff88209075cad638ca0ed6662f499c74bdd3a256d3511b9fe
11f95734f602e7d1481f9887ca7fc8bed83258e22fd5c443449ac159a4732115
9a199388f2e2bd43deac789cd248363bdcb32611480ac27cf62420b4c351759e
a908da1d873928f9f872dddd33c29620dfa0a54646b79bfde672bd15e6986ec9
2a82215ea19fcbded36fa95df35b6e7c5f6ed28b0b5f6c3460b4223f1904a536
2c17f46f8323e8e692a1827bb18c9a9a43fcff9860b72372cb9551b7cdd532f5
d805a5c9f398cf5126ee84e6899cc9a3d34444a7e2337d15cdf9cc2a949649ba
f4fdca7e355eaf9362449ebf7df78a366e239091bdb4c4a34b9dfcf9882f23de
527d5f9f9cf27824b84109a495eeced7c7c097fb324b872212f6813a660e5ee4
aba8388e52d25b4006e78d4010c26cf8848c3e3145b0c050b9d55c8aff211fe8
95b0921a4de64f6e510ac36b5426ecea550c4d3a6033dfbd0d93dd2213fea714
f22ff1f6c067d0e085eb79fe5c2e35df01f9edb2bcc73e59e1e15eefd32b8f07
6df4693c156b7581ffb9a1267a84c055bc9edd8170681b9021bb837f1e9d1167
1f6132e867a52e5c7b084011484f7fcfec9195f407c9bfb06a3f13f5b709d333
6dfb6aade6429e2cca0718a497a442299a57199dfd547813471f4cf30ed6c7c7
c6675bf6647188f84ee445590a494a8d516635bad77ace98ec61d28746839a8d
2c5982946e20e2a85531ac00121697b263591e055ac19586db59a890fdaf98b1
0499deda2f068008d488551abf96b4b758c6ed6b79cd2ec6a204d1250b140421
402e56855bea7ba7f63c4dfd700c95432efeb626c6d303e4e51d5d4c9557db6a
c976648d528b54097ea84178f265a6efb5ba941f1b8cd00008235675f1e9bcf2
a357d96731893796391aec9a4daafc89c857d94e72f7217eaeb820e2a9bd6a47
16fc1a491a2f9711a014f639c65a78c4cb5c7a568da179d4ccbccfb1fcfc2d24
ba4e9d1ab1f79dee69a26796cf701e83a97433ad08814b30191bbd4600860edc
c3043e58df607cf3448404c3f21bcf3540afc3283eb31bf9903b9b4dadef2921
54bd80737f1e5f1e0c43af7101b66ad883ed48c79dd719d75f082791a64dadc5
22a38bfb413b5947af60a90709f4612bd09fc3701d4051e0fcc8fb004efa642d
cda8b1de7e358a20f72671af0b08098086afb6e3392ea273350d9bae22bbd87c
390df766dea6530de723fbfa348d43441128d2e03e39d75591791c7d39186120
40c680dba7c76b7fc953156dc915d0a9d82582dd66bb8ba5f68f42d1ddd4e486
389abb8f40fc2e68f379348af87c9abfd506bfcf506bdcbc18f0c0ad3b238e85
ef02cc70ee34067fc52e0f36c57b5509ac4ea3c659fd2463f08f1e101f252ea6
c22aba3cd3d168560febcd042f56de61f6d02d593d74c77b5beecb89791f1268
253a180a13ca7208af286f9fbc80295acf290ce1606af1b4e19a5e304467c967
cee97a8a3330c9c407b3cbf8ee4b3bab06d4294e08a7e6b9a7d64d7f07ed97ca
08c98c31097ac2d060c40d72045738c275e9feaab86974666e87557c1b86fb30
ef48e542b5544e5ad49c167983055b8eebd48b77d94dd19a8c408c3fb894bc10
7994355f8431622faebca90825423f38be418071bf1e859aaafd76f2a3dddaf2
f0fe71ed7935e959c2478d1790e02cd8dfd981f12506795f4b33ef44ec1b40ff
09dc173d887e37c39cef2242d0da8491e0c566db012ff8af6ec5443924a03f8e
a1cccb6363eb69b14be1666c10b4e6f3626d7f7f39d101222b9d1c50a8fe50ed
94364865e2a3f0a22e639bc212a0241386415cc60e5fd2106d029ca23f997bf4
a9822a81de53b287b11ccab159be018e1085c75e8cf61640d5384e471451ee0f
57cffb227c0ea9275cbe8eb7a8287b2cf77c79591c79c4d67be5efae0fdeb480
4342367db683238b220484295aef67f84b6b4f0eb4df000be5c0519cac77698b
27f1402e1642836d12b3b5ed58678f9874dca9d71ceca421302bd5e1d940658f
2bf39d2a017c631acb9635c4dcf9666dd4ae647119d877deab34a5b3ff94b869
aa2a15208cad319abd3738bee4e9f7ceaffbf4d56dd7eeff090c442a584ee1f4
6b236c5d2b140ad4f5f89a7bab66558f544ca5dbc56712657c189ea0d3596c19
3cdee33a05b94777870a69289db8a289e4ac5e7da79c4b6c915f0eed46e5ab59
16e7a9bacba52188ecc9251cfe08ada7754088e00d8b9c099626ed84e41b3cda
fabb6a2496aa51e1205bc454945ca80b2bc1e99ebb9e21d746c3f124db4e2e82
2c6778f2b9ecdbdab786f603148424b35498ae3096f7e8118d23a5327130416e
0a3e062e9ac1b71fa3e72e0ebbcbbe15ba5a998827697b2ac9663753ae931aac
c43fa2cd286b9fde666f0805ea1ad8fcc46667e629c88af962357ffb6b35de50
c3f6b7718cc440106b768588cd530da88a34d226ee584cc356d1d9e9cd769e3a
2753cfaab5930d988ddbd7846c8d52031dd16dbd0fea15986bca800b2166711f
0036922c19213c53a71203a1c4f9b42892eaf0a545e85899771a98142732d7bc
9aba49ea3701ec6cfe753dc8071b5e487d575b5f04e236e6cba9aecedf271f11
cebece7f5ebd4d85c947382382416dc710ded67276de6f96ab9b4d5d242c9dc4
c71563a982aa8ce77ac79c1dfe087c1d70c2dea295bd70d5f80e56ca033ddf64
76a560baa3e4a778f321033854ff4e1c22333fce2942359c7534950ad6e38e38
63608f487c9f22541c1f8778ae6771658425d60d1c0819e1713d9bd3b0ce9252
c743a39ae96d6e3df5c219804d77c75c4d04e58acd9e2db4c26e9952ebdab48d
3e852f69528aa2be018ee0d76461a3afcdfe5dcab74b9f2abf93bd1342c6f52d
05db4067b068c41f0b33744b96818a21b35eacdbd3ea11d3bbc77fa36aa93b6f
9de987b7e2739f333b5225cee7b799461bc4697f2db715601a5f4cf5bd9af7cc
79fa8737d5b95f45c35b0c8a87b20c5a1eb73095eb32fee208e13de824d59d76
799e4b2652eeb01424079c0e1e272609e0e564794c7a6af0ebd81ce075c3fc68
017a695f2234c3e36c6fc73d6764b0f713ec9bdedb07f021d63f59a5d2c07181
75c1c68d2a38e7013b562fd74dd91a1d2b8d8e83303fdd6f7dc55686487101e6
b40b266b5cc010ecdcc65fe95c3c708e3290e86e7380680ec31100af394c7065
8f3f8e01ff3c82ee28cf0f4b69df5b662d0bd40b4350c7faea3555096072ae19
ab5fc6bd9234378b673d27db7059cdd7fc728ca444ea738812097763534bfe63
5140fc51893cf05f9f03842e4b318fdafeb6e4636afb6a03bb79165bb153e100
b241aac6ed7f653107d3c9bb1e5cad42495fcaf7dc530316c4fff751471a4d65
4dc5a69ba4f2038bd73182b69e13d3669a77bfdc5fdaf8e41e615fafc51eb359
dc1e7754534de44adc73fb52a5bb8669fe2828e61e0069b834a8a6942ad952c5
4eeb28570e1ce09b67dc97b80060ba084ed524eab3f6356911288cfd3833682b
4ef450191d74d0e7622b24ba12392d8ed0846a502cef3d4db5992d19f8789eaf
dc61514611f3e66224bed21b2c53594e61057805ea6a429bad3b0f903ea8aad0
dcd453878def23546849af1ff2d3d6e538acf2d233f9106753e15f1ed3f4f4b6
517cd7f7578a66ffb1db890e4e9eb82575bba1846237331117874aae1b144f1f
75ee28de242ec3fdb73c28ba2b23ada96c865764210fee43dc68a0667295effe
9d3119b26b6fbbfe9e367826c2b397fbd42833e9c242e88f9ca06f4d5dbc3240
c2845f454270252a71d8fb42910485cbfb1ff5e3c53a077ed877736d8d231547
0ff747e5b47f102114bc785df90ad4fa752ee1b1bf74cbd17c7b580755cb56c1
1876935a385c9070d25204e4cca86829fc15d6a5ab602b1dc48d4bd33988ac11
ed72fb1ead284e7f16d6ba2547f69a7ce33fff6e47ab8ce2e8b4236e4beced6f
39d0bd6f9071f74e309c574b960c68c26f23da6b18baf7ae8715dff59903a061
504e4f0317a2bb481c7878ee625c1686135211da8df19744d2b78c9598f13761
b33a5450070dc69e175a6f058b544e2428b76a649cccc819957796b5fe5ec1d2
aa2010ecad86adbae918da3d5e90d8ad452b30dc2e32bde8dd6d0d4a6b02b667
5f206ce247321fab33b6c1674a9ad5f9e1955b7dc6ae5104f40b56274e412593
fb45eb5248b632b5e95637a9aa380d23f7d164472ade18daac4efa033cabd512
ce3ea859fc950bc8fe9b762bcdffd4e6d56e92d22c8962f9e9d2fc69833d687d
071c68cbb1afef46565f2dcfad6df4a077cb7c54fb22322c8ff65057657d2017
fa4e47d5ff4bd1342911076839d96ba67b0f72552ec574f17b7574ec7ef28fed
58a7cb59a3183f7cf4223b926070e9a2e0059d16ec8d897348047ac743877c8f
a622715b01d26810c6bcd92e4055d376024f1e4906e5809bb45b90eedbe483fa
268674b22729dc42a7408372d3f94c8a587ac01171fb9164f2a5ebd589a2b4b9
c75db27aae25088504b3f787d14189b555110a78bad9f8cd5c988b2dedbeb13b
632d64289534539bfb811186505d301b98fb6d8ff9bbe03fa753c6e0b38a58ea
d27740f4621f3330ab94b3070798bb5ef0b242a6c41f476ca8033806b979411d
8a2553a1a9daf028d7deb9f86f4d3b1542e8ab957bcc5de58b5b1c3ccefa4f7b
19241950caa11abbd7605583764dae33694742389a8b602cb553bc8276c63e55
f4942a722c5d82b6a6c0cf8122d600af4c88105f3f7416e48c2c99ebdeecaed5
88ea82c0cdb3a6657c995d2b096c55272cba856acd2951f7cb92ef8ea533d1cf
9c9e8d0f97ae449d799793e32347eb83ef6a743e2935bac5e4458b58d79d09f4
01540b9b0b068942c8adeec17f825f0a9a3b9f52a1f61704bfebbb4c406064a7
a84e6c888591b8182c32e6ed18291d6eaf9eee366fa818b014e9da41028f9576
d0c980aebcd77906165ddd14c8257261266bdf444378f3e881f7d8ed76ad2a56
646be9b27b686ec9bbe24c575292a653b8ed6dddde0ea5d79a462c314fe960c7
a55a888455f193f6d48204519cb0568e01dbc30d23e76cb2f2392de6ff2d0a4a
356fa940d354284f3dcac45220910cdec99046131b8e3123de91f43ed2839bbf
a191e799d6ca2258faa9f4cfe3d9a55317c96d32c92ab8eba638a888b809b6be
c16e1e2d181fc4d4bb9b88a1b32d44b3ab54aaeb1b76ac46951920f501c087e4
7597e564b5c500bd2979e29e6b130437d089570148a8d5a5a4c2cf70e283964b
3fa2ad109d9ea27936ac3c09c9fefb055d67ca3598f2cce1889aa6bdd4f33691
2cd412daa3383147d43cd0151c66909377d6c8fbe3b2909e82a59079eb255bb2
0dfb5bb59ec267ceecdc0eef5c7c51f3e929bc4ef373c01f864857123a41e737
7d14894064f8fd94e49541576e5e5ab0a230448d6b73a32aff68d1e9eee4f297
2f5063d47738992b33fd36c78ebfa8d3e35932b4fe30dcd8207fb811756d15e1
b55ba6dec9689e5631c3556f81602a87536358c5c2213a682dc5714213291d16
6fe242309c5794ae07287ed143fc9597ab6a0287ddefa54dfdc336a851ca91f3
07a0d7af7b149f35d0441e3b732fc4706ac286fca96748d7c3e4ceb95af46558
0ae7ae87b394fd2199a6d3979637650610c73f74767bd64fc58acaf711efe061
a981ef21419542b41e091619ed3e00ee415d35dd446880c3388bb90d7e4b213d
d2e46052124df23ea64a74362089be543ca644852b3e84f474d216efb9283c3c
63caf58f2145fb688623de0d064c3d8e52cb22f78ba7ab73fea9a83119475845
43e8654538256cb8a54b6d623fef6fde3fc1ac280506897419151da8ed65e9a4
c1739e8996cf285676df61bd290b2c1633d5d87423c56f95c172a3294980a988
2876cd43144425ac3d7dc1c282eca9f3a4ce2b69e9de2612293d1c2f0c6d4fb1
579954efa118b0ee176cc7f051b7cf31e567dc89fe10db5267324c579cb701bb
c3dde5118315216fd90a1639953dbb90d4d3fe3a4ec88f390d0eefb07502328f
7d5ba47cbb56e55a6e8f4c4e5363ff5dba943dde5bbe9a9e700457c573a16ede
ec4e575eda90934117c3df1fb803a6d506b8a237d95a9b03cae83cb01e8ac67c
9073c880145cdc8bdab42242c95f737da35c6479240b8de28b61a9b37faed10d
96bf60de6944396a3ac549a25a2c5cb2cbcd1f30eec8203fb96e29622e14b711
daec5ea23568cb5d9550d9dfcef80efe375961a47c197ac4be5cee9612d364f9
227147e198205bb2e251c982a389c7f3e7c1bedc81feec84913b53c7a89cbd44
a3042049ff38a26d648871118b2e6411e0df7fd80e18ae27a6488453702e7f70
2370040c3005560207aa682a15bde323546a34f9d273d60660200a714f485d0d
131be4afa5e50134e9e973c493b0d96aa5dfff42dc646735cd7f5ae8c87afbc7
417aea0ce42cf632c66302e700bfda3c37d791f16befa6d8f26ef5656c9add75
4a5621dd33128bb13772ee6bdca2d9530fb02792b4955121d175129fb86cf456
da8d8cf796bb05ab165bf9cf4843a9fd569a1ef0b25d600450b0c9835c906fd6
2ae5b0882dde3115733d179aebc05ae4c0578d11ab777cd71bd05714f6a11a9b
9e1b5216a0818236ba2bc9f821e73756937dcc0af19c8ae9316e17932b393c4e
d74dda135d0f1fbb88406de8b061f894145b75db1304a46c4e707ea2939caed1
7d8149053aebcd8e8ee2e06aa1e0c51f7c3a8ae895e149a026dc7595c7a1ef35
452ffe5dc1dc74b9e127fafd38ebf90413015d35c3daa19fbb50687cd6596382
d7e67139dfed277636eb60654fc567dbc62de91c53317909b8c7f5e8f1e03f68
ef0483f5de00f8c7f9e0a79d2f1cc8958478ba06e1163792d37fe2e12f14f6ae
1709fd106660c9c52870fe7a91a8307224d3b45a4d42002967b99dc87f2c11de
e7d9d06a0c8d6f0d8d219e3f265499f52077dd8a15b8eb240664939059392777
dc035306b43327d85c2c54dbf8d32fc1feb4addb472030e59bed020722b48d02
ec421cd9de8107d72307827e97465550257618ab34db921537de47081cbd04b6
bd36e8c846ae88f3171a68d56c4ea599d7805f138db7b0e26835cf1ab8f6dee1
01fa9af2dad55a3be29d1fe93f385ad2acade52ed71c6af3dd6f65c9da7ba142
f3634e7baefda6df6a7a764876d0eaecef9b590d59f0ae1207fb3a99f68c51b2
590f9e1b299283409cdf91f81529eb0d2325aa9ec4fbe3272740af99620f0024
b3f83f6c5460f4ba9a21779bc314df3c18cdbd2c4b0c75bbb22d40859f8a6420
88c92057225041e6576c2a623b893421869db554dd671ce57e984c4ab8fa2c88
f132f302743f83ae7ce9ed1be6017917173ca431c52abc6336fa41b1e027cd59
5cd9dc2d52b79df23bf9cd005a95ab484acd67e31e9c38266bdc863278d64cc9
b90d2cf2c33dcc9405168eeeebc3107d721faa3140dbd890aab6d1f2afd2cef8
e2c09b2eba4ab690712f893f5f85ae6b4fe4848bfed07119b4280ccab1e36068
bf83a69a4bcec53ac5c3e39bcb45141b6f26f076429dfab05b222abf86f17ed1
30c13a445ddafa6093c665706fd6ad713b94a2bc07bd580b9b2c97b259da2d4b
27a247790a1521877d1745861da74cb99a7d8e430be4a237b95140453b9c2eeb
f45d8f033ea698bc50ee73e2dd0358b58d81cbd8ecf3011299dc88254dc8ae21
d4ce3d47e23c8133ed3b42d683e0f6a2c60c574b408e6747431c9a0b199ff733
fa4ac43f853aecd189647b25fa4a1b9098d759579ed39202c46d0c835e96cf0b
4ea921ae2fc435b17335f485b3dac0f53555130d876c862cf52c560476418f3c
f1c378262aa31437185bd52189bc9793ba5ea6774789a34c9e658e85ee4ebf6a
d7eadc915dae28172fffa71b7504b7ffa2409b09783cc8cf6b7fd3e1d829af3f
237893b70dd8ba741a256b14f5e8557e23f38643a12c3ffd29e9e9af3136f737
2c8d111408d8f116bcf5b2994c2ea2d810e9810363943c385594d1ab8668b78e
2cf0c39a1a6c55addb52232a77e7f714f0b1bac731c614f48baeb9cebde726c5
080b4d03294a1de52d81d69ceeb61a54cb510da3eab70a73721faf32e0406ff3
5d047d1a07c96163a9341d6be658de0183ceb8299d34818607da21323eab41d9
f86c7ae3888bbc4c3432d00f89c63dd74eb5764332a64cabe37184c0ca435965
a7076c4f5c58e9e81f7ef76e3fbf6ec7dd28f5fd9d644500f0d007cca4fd3367
da81a32c7c15ff1ee6bef20eb354066a9a873277131c5ec065cb5082e03c71bc
fd34b7ecb0caa3bd511967093b5d7946b2d0206e1496d021ea7b1a4d908ad557
c604405c61a4af20f0b0fd128848d4a0aef09d9678fa7da71c576c5dfe66a6bb
6dc64d3890685a662e59cf870fff36f265f15e46d189d3c4feafb0f76d2ba224
b790b0092a39d1144336e52e36c156e7b6c2c08f41facad3cc5cbfc63c509422
4b65576e7570cec86d6abb988889fb71304f60a49ae08de443410b0aaa44f55b
1ce71e5cf0fa5fd2055d0a786be91fc4a6edccb4f9114ea29ff90563a35dd2dc
457dc1ce3126f4707cd06e29114df0625a4fc69051c5c41c1dd2d2853198d078
3c2d7a838127de586313c63d7867224a677249a9576b957152ce609124de23d4
76513dc871c41174e02fa4e936b79b7f3293298423aca222ac96ed78db049e31
059c609b46f7c681782c32d9a97514b357d3f468ab9cdffab45631e18aac9059
141446ec0e41f16e627c8e9ec237758d981866c0c39f89cbe41903d9dd1aadac
613bcfa8be07fa28ab064d1b9a1ecf1940630e36ec1574868c1d44ea6901fb93
a9e574313b614eec19bcc03d6c3f8e44deaf5b866cee7080bb239e415a189d15
f67b8e330359c94e96643623af08f9eed551375f8306b54d32104614c0906716
237421810b6e6cbfebd1d5fa853413e0e881261cd64991667eddb01996caea7a
ed177c35997892fd28292df42bfa69dcdb449cc62a89e42c45db24495a39283a
bca7b428abb8631a21dfc630b7582b7b048e87391ba46463726340fc91198561
9e69bcbbbfdd17229a4b0897e20d66f4f4f22452efc9dcca75ba21c03b90ec09
0850642fb0e65819176317438122e4ae23fee2ed44f3758ce653cdb944b947c3
9f1e08e09cb153db9c348cb9b42b199b076f70ab6725140389ab1947e2134c16
f72373f17cf3de3624c8c102bf428e0bb9ea662b5dab86f19ddf03d92913c83f
1f7c4e8e641d24e76abfa03615726236aada72f878a08014c9fa4f4cc3142cd9
0eda6ea2c51124fa08ba73dd0ca455efc1cd3be03e7a374d0a5f03982ef6c47b
81f6df2fcbefbd9da4e06e0f7923194607cc834fd584b061f083af99342b8376
28614eb08dedcefd241feeccc171f294b886d1a1fe4253e0b047d5b5e3214dda
557da318a2322b7ad3c40053ccd03ce2375e469c47a1b6e86ea3de5cee15e2be
e7fe73646b2d1c6d260553e1a717cc32785e3167f8cc4ab68a3607e5cb643514
a430c98bb36dc61f96279cb541bc7a20fe3843f9d057c53e2291ea80e19ebd23
172a3ebc82eb26ef5ab3602c34d632265b608c43ae971768486b66f2c79cbf13
813a2590ad5565a94924a64f20b51eb7b718954a970f422929df7bed73238167
""".split())


class Report:
    """Collects problems so that every check runs before anything is printed."""

    def __init__(self) -> None:
        self.errors: list = []
        self.warnings: list = []
        self.violations: list = []

    def error(self, message: str) -> None:
        self.errors.append(message)

    def warn(self, message: str) -> None:
        self.warnings.append(message)

    def violation(self, message: str) -> None:
        """A file the grader can read, but that breaks a rule."""
        self.violations.append(message)


def _check_meta(meta, r: Report) -> None:
    if not isinstance(meta, dict):
        r.error("submission_meta がオブジェクトではありません")
        return
    for key in REQUIRED_META:
        if key not in meta:
            r.error("submission_meta に %s がありません" % key)
    if meta.get("path") not in PATHS and "path" in meta:
        r.error("submission_meta.path が %s のいずれでもありません: %r"
                % (" / ".join(PATHS), meta.get("path")))
    if meta.get("recipe") not in RECIPES and "recipe" in meta:
        r.error("submission_meta.recipe が %s のいずれでもありません: %r"
                % (" / ".join(RECIPES), meta.get("recipe")))
    if "base_model" in meta and meta.get("base_model") != EXPECTED_BASE_MODEL:
        r.warn("submission_meta.base_model が %s ではありません: %r"
               % (EXPECTED_BASE_MODEL, meta.get("base_model")))
    _check_chat_template(meta, r)


def _check_chat_template(meta, r: Report) -> None:
    """Check the chat template framing recorded by the inference notebook."""
    info = meta.get("chat_template")
    if not isinstance(info, dict):
        r.error("submission_meta に chat_template がありません。"
                "配布された推論用コードで出力し直してください")
        return

    if not info.get("applied"):
        # Route (iii): the Base model has no chat template, the question goes in as is.
        if info.get("framing_prefix") or info.get("framing_suffix"):
            r.error("chat_template.applied が false なのに枠組みが記録されています")
        return

    variants = info.get("framing_variants")
    if variants != 1:
        r.violation("chat template が問題ごとに異なる枠組みを生成しています (%r 種類)。"
                    "全問で同じ枠組みである必要があります" % (variants,))
        return

    prefix = info.get("framing_prefix")
    suffix = info.get("framing_suffix")
    if not isinstance(prefix, str) or not isinstance(suffix, str):
        r.error("chat_template の framing_prefix / framing_suffix が文字列ではありません")
        return

    if (prefix, suffix) == REFERENCE_FRAMING:
        return

    total = len(prefix) + len(suffix)
    if total > MAX_FRAMING_CHARS:
        r.violation("chat template の枠組みが %d 文字あります (上限 %d 文字)。"
                    "問題文の前後に入れてよいのは役割名と区切りだけです"
                    % (total, MAX_FRAMING_CHARS))

    bad = [w for w in framing_words(prefix) + framing_words(suffix)
           if w.lower() not in ALLOWED_FRAMING_WORDS]
    if bad:
        shown = ", ".join(sorted(set(bad))[:8])
        r.violation("chat template の枠組みに指示や例文と思われる語が含まれています: %s。"
                    "枠組みに入れてよいのは特殊トークンと役割名と空白のみです" % shown)


def _check_predictions(preds, r: Report) -> None:
    if not isinstance(preds, list):
        r.error("predictions が配列ではありません")
        return
    if len(preds) != len(QUESTION_IDS):
        r.error("predictions の件数が %d 件です (正しくは %d 件)"
                % (len(preds), len(QUESTION_IDS)))

    seen, empty_answers, bad_rows = set(), 0, 0
    for i, p in enumerate(preds):
        if not isinstance(p, dict):
            bad_rows += 1
            continue
        for key, want in REQUIRED_PREDICTION.items():
            if key not in p:
                if bad_rows < 5:
                    r.error("predictions[%d] に %s がありません" % (i, key))
                bad_rows += 1
            elif not isinstance(p[key], want) or isinstance(p[key], bool):
                if bad_rows < 5:
                    r.error("predictions[%d] の %s の型が %s ではありません"
                            % (i, key, want.__name__))
                bad_rows += 1
        task = p.get("task")
        if task is not None and task not in TASKS:
            r.error("predictions[%d] の task が %s のいずれでもありません: %r"
                    % (i, " / ".join(TASKS), task))
        qid = p.get("question_id")
        if isinstance(qid, str):
            if qid in seen:
                r.error("question_id が重複しています: %s" % qid[:12])
            seen.add(qid)
        if isinstance(p.get("answer_text"), str) and not p["answer_text"].strip():
            empty_answers += 1
        for field in SCORE_FIELDS:
            if field in p:
                r.warn("predictions[%d] に採点結果らしき項目 %s があります。"
                       "提出するのは推論結果のみです" % (i, field))
                break
    if bad_rows > 5:
        r.error("... 他にも必須項目の欠落・型違いが %d 件あります" % (bad_rows - 5))

    missing = QUESTION_IDS - seen
    extra = seen - QUESTION_IDS
    if missing:
        r.error("評価用データの問題のうち %d 件が提出に含まれていません" % len(missing))
    if extra:
        r.error("評価用データに無い question_id が %d 件含まれています" % len(extra))
    if empty_answers:
        r.warn("解答が空の問題が %d 件あります (採点では 0 点になります)" % empty_answers)


def check(path: str) -> Report:
    r = Report()
    if not os.path.exists(path):
        r.error("ファイルが見つかりません: %s" % path)
        return r
    try:
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
    except json.JSONDecodeError as exc:
        r.error("JSON として読み込めません (%d 行目付近): %s" % (exc.lineno, exc.msg))
        return r
    except OSError as exc:
        r.error("ファイルを読めません: %s" % exc)
        return r

    if not isinstance(data, dict):
        r.error("JSON の最上位がオブジェクトではありません")
        return r
    for field in SCORE_FIELDS:
        if field in data:
            r.warn("最上位に採点結果らしき項目 %s があります。提出するのは推論結果のみです" % field)
    if "submission_meta" not in data:
        r.error("submission_meta がありません")
    else:
        _check_meta(data["submission_meta"], r)
    if "predictions" not in data:
        r.error("predictions がありません")
    else:
        _check_predictions(data["predictions"], r)
    return r


def check_and_report(path: str) -> None:
    """Run the checks and print the result inside the notebook."""
    r = check(path)
    print("提出ファイル: %s" % path)
    print("-" * 60)
    for message in r.errors:
        print("  [エラー]     " + message)
    for message in r.violations:
        print("  [ルール違反] " + message)
    for message in r.warnings:
        print("  [注意]       " + message)
    print("-" * 60)
    if r.errors:
        print("この形式では採点できません。エラー %d 件を直してから提出してください。" % len(r.errors))
    elif r.violations:
        print("採点可能な形式ですが、ルールに違反する箇所が %d 件あります。"
              "このまま提出すると失格判定の対象になります。" % len(r.violations))
    else:
        print("採点可能な形式です。")
    print("※ このチェックは形式のみを見ています。正誤やスコアは判定していません。")


# 保存セルと同じ規則で提出ファイルのパスを組み立てる
SUBMISSION_PATH = PREDICTIONS_DIR + "/" + RUN_ID + ".json"
check_and_report(SUBMISSION_PATH)


## Step 8: 提出

**提出するファイル**: `{PREDICTIONS_DIR}/{RUN_ID}.json`

このファイルを omnicampus に提出すると自動採点され、リーダーボードに順位が反映される。ただし、参加の可否は過去の提出物全ての最高点で判断される。
提出手順は運営チームから案内される。

**提出にあたっての注意**:
- 提出期間中は何度でも上書き提出できるが、**有効になるのは最後に提出したもの**。
  良いスコアが出たあとに別の設定を試して提出すると、その結果で上書きされる
- 手元で採点する手段は配布していない。スコアはリーダーボードで確認する
- 推論結果の JSON とは別に、**開発モデル (adapter) の Hugging Face への
  アップロード**も必要 (下の Step 9)

**local disk の掃除**: `/content/sft_work/merged_final` (~8GB) は session 終了で自動破棄されるため
明示削除は不要 (残 disk を節約したい場合: `!rm -rf /content/sft_work/merged_final`)


## Step 9: 開発モデル (adapter) の提出

提出する開発モデルは **Stage 1 / Stage 2 の adapter 2 つ**。期限までに Hugging Face に
アップロードする。アップロード時点の公開・非公開は任意だが、運営の要請があれば速やかに
公開すること。

下のセルは、推論に使った chat template を adapter に書き戻して、アップロードするものが
推論時の状態と一致するようにする。


In [ ]:
# [adapter-finalize]
# 提出する開発モデル (adapter) を推論時の状態に揃える。
# adapter に入っている chat_template は notebook1_sft を実行した時点のものなので、
# その後に推論用 config を変えた場合、adapter と推論に使った template が食い違う。
# ここで推論に使った template を adapter に書き戻す。
if PATH_CHOICE == "iii":
    print("[adapter] ルート iii は adapter がないため skip")
    print("提出する開発モデルは " + BASE_MODEL + " そのものです。学習していないため")
    print("Hugging Face へのアップロードは不要です (運営から指示があった場合は従ってください)。")
else:
    from transformers import AutoTokenizer

    for _label, _dir in [("Stage 1", STAGE1_ADAPTER_DIR), ("Stage 2", STAGE2_ADAPTER_DIR)]:
        _tok = AutoTokenizer.from_pretrained(_dir)
        if getattr(_tok, "chat_template", None) == CHAT_TEMPLATE:
            print("[adapter] " + _label + ": chat_template は推論時と一致 (変更なし)")
            continue
        _tok.chat_template = CHAT_TEMPLATE
        _tok.save_pretrained(_dir)
        print("[adapter] " + _label + ": chat_template を推論時のものに更新 -> " + _dir)

    print()
    print("提出する開発モデルはこの adapter 2 つです:")
    print("  Stage 1: " + STAGE1_ADAPTER_DIR)
    print("  Stage 2: " + STAGE2_ADAPTER_DIR)
    print("期限までに Hugging Face にアップロードしてください。")
    print("notebook1_sft を開き、config セルで ADAPTER_STORAGE=\"hf_hub\" にしてから")
    print("config / adapter-push の 2 セルだけを実行すれば、学習をやり直さずに上げられます。")
    if ADAPTER_SOURCE == "hf_hub":
        print()
        print("※ ADAPTER_SOURCE=\"hf_hub\" のため、更新したのは HF から取得した複製です。")
        print("  Hugging Face 上のリポジトリにも上げ直してください。")
